# Checking the isolation of subinterpreters

### Test 1: Isolation of globals(), sys.modules and built-ins

In [12]:
!python3.14 ./Subinterpreters_check/isolated_states.py

Test 1.1: globals() isolation
main_interp: False
interp1: True
interp2: False

Test 1.2: sys.modules isolation
main_interp: False
interp1: True
interp2: False

Test 1.3: built-ins isolation
main_interp: 6
interp1: -1
interp2: 6


### Test 2: Shared and isolated objects

In [13]:
!python3.14 ./Subinterpreters_check/isolated_objects.py

Test 2.1: non-shareable list (mutable object)


main_interp: [1, 2, 3, 4]
subinterp: [1, 2, 3, 4, 5]
main_interp: [1, 2, 3, 4, 10]
subinterp: [1, 2, 3, 4, 5]

Test 2.2: shareable memoryview
main_interp: 113 q
main_interp: 65 A
subinterp: 65 A


# Comparison: Subinterpreters vs noGIL vs Multiprocess

In [1]:
import pyperf
import plotly.graph_objects as go
import numpy as np

In [2]:
def get_result_data(filenale: str):
    suite = pyperf.BenchmarkSuite.load(filenale)

    data = dict()

    for b in suite:
        name = b.get_name()
        values = np.array(b.get_values(), dtype=float)

        mean = values.mean()
        std = values.std(ddof=1)

        data[name] = (mean, std)

    return data

### Test 1: Start-Stop

In [2]:
!python3.14t ./Tests/start_join.py --quiet --rigorous -o ./Data/start_join_nogil.json

threads: Mean +- std dev: 383 us +- 7 us
multiprocess: Mean +- std dev: 162 ms +- 6 ms
subinterp: Mean +- std dev: 19.6 ms +- 0.7 ms


In [8]:
!python3.14 ./Tests/start_join.py --quiet --rigorous -o ./Data/start_join_gil.json

threads: Mean +- std dev: 155 us +- 3 us
multiprocess: Mean +- std dev: 134 ms +- 5 ms
subinterp: Mean +- std dev: 12.4 ms +- 0.2 ms


For the purity of the experiment, only threads will be taken into account from the free-threaded python version (noGIL)

In [45]:
d_nogil = get_result_data("./Data/start_join_nogil.json")
d_gil = get_result_data("./Data/start_join_gil.json")

names = ["Thread noGIL", "Thread with GIL", "Subinterpreter", "Multiprocess"]

data_mean = np.array([d_nogil["threads"][0], d_gil["threads"][0], d_gil["subinterp"][0], d_gil["multiprocess"][0]]) * 1000.0
data_err = np.array([d_nogil["threads"][1], d_gil["threads"][1], d_gil["subinterp"][1], d_gil["multiprocess"][1]]) * 1000.0

In [53]:
fig = go.Figure()

fig.add_bar(
    x=names,
    y=data_mean,
    error_y=dict(type="data", array=data_err, visible=True),
    width=0.25,
    textposition="outside"
)

annotations = list()

for idx in range(2):
    annotations.append(dict(
        x=names[idx],
        y=data_mean[idx],
        text=f"{data_mean[idx]*1000.0:.1f} μs ± {data_err[idx]*1000.0:.1f} μs",
        showarrow=False,
        yshift=22,
        font=dict(size=16)
    ))

fig.update_layout(annotations=annotations)

fig.update_layout(
    width=1200,
    xaxis=dict(tickfont=dict(size=18)),
    yaxis_title="Time (ms)",
    barmode="group",
    bargap=0.5,
    legend_title_text="Method",
    template="plotly_white"
)

fig.show()

### Test 2: Inter-Worker (Ping-Pong test)

In [2]:
!python3.14t ./Tests/ping_pong.py --quiet --rigorous -o ./Data/ping_pong_nogil.json

pingpong_threads_1b: Mean +- std dev: 43.9 us +- 3.6 us
pingpong_multiprocess_1b: Mean +- std dev: 177 us +- 14 us
pingpong_subinterp_1b: Mean +- std dev: 12.5 ms +- 2.0 ms
pingpong_threads_1000b: Mean +- std dev: 42.5 us +- 3.3 us
pingpong_multiprocess_1000b: Mean +- std dev: 169 us +- 10 us
pingpong_subinterp_1000b: Mean +- std dev: 12.6 ms +- 2.0 ms
pingpong_threads_100000b: Mean +- std dev: 41.8 us +- 2.0 us
pingpong_multiprocess_100000b: Mean +- std dev: 443 us +- 16 us
pingpong_subinterp_100000b: Mean +- std dev: 13.2 ms +- 2.4 ms
pingpong_threads_1000000b: Mean +- std dev: 41.5 us +- 1.7 us
pingpong_multiprocess_1000000b: Mean +- std dev: 3.34 ms +- 0.18 ms
pingpong_subinterp_1000000b: Mean +- std dev: 12.4 ms +- 1.7 ms


In [3]:
!python3.14 ./Tests/ping_pong.py --quiet --rigorous -o ./Data/ping_pong_gil.json

pingpong_threads_1b: Mean +- std dev: 36.2 us +- 2.7 us
pingpong_multiprocess_1b: Mean +- std dev: 137 us +- 6 us
pingpong_subinterp_1b: Mean +- std dev: 11.1 ms +- 1.8 ms
pingpong_threads_1000b: Mean +- std dev: 37.7 us +- 3.2 us
pingpong_multiprocess_1000b: Mean +- std dev: 140 us +- 7 us
pingpong_subinterp_1000b: Mean +- std dev: 10.9 ms +- 1.6 ms
pingpong_threads_100000b: Mean +- std dev: 36.0 us +- 2.5 us
pingpong_multiprocess_100000b: Mean +- std dev: 331 us +- 63 us
pingpong_subinterp_100000b: Mean +- std dev: 10.8 ms +- 1.6 ms
pingpong_threads_1000000b: Mean +- std dev: 37.1 us +- 1.4 us
pingpong_multiprocess_1000000b: Mean +- std dev: 2.36 ms +- 0.19 ms
pingpong_subinterp_1000000b: Mean +- std dev: 11.8 ms +- 2.3 ms


In [47]:
msg_sizes = ["1", "1K", "100K", "1M"]

d_nogil = get_result_data("./Data/ping_pong_nogil.json")
d_gil = get_result_data("./Data/ping_pong_gil.json")

nogil_mean = [d_nogil[f"pingpong_threads_{num_bytes}b"][0] * 1000.0 for num_bytes in [1, 1000, 100000, 1000000]]
gil_mean = [d_gil[f"pingpong_threads_{num_bytes}b"][0] * 1000.0 for num_bytes in [1, 1000, 100000, 1000000]]
subint_mean = [d_gil[f"pingpong_subinterp_{num_bytes}b"][0] * 1000.0 for num_bytes in [1, 1000, 100000, 1000000]]
mp_mean = [d_gil[f"pingpong_multiprocess_{num_bytes}b"][0] * 1000.0 for num_bytes in [1, 1000, 100000, 1000000]]

nogil_err = [d_nogil[f"pingpong_threads_{num_bytes}b"][1] * 1000.0 for num_bytes in [1, 1000, 100000, 1000000]]
gil_err = [d_gil[f"pingpong_threads_{num_bytes}b"][1] * 1000.0 for num_bytes in [1, 1000, 100000, 1000000]]
subint_err = [d_gil[f"pingpong_subinterp_{num_bytes}b"][1] * 1000.0 for num_bytes in [1, 1000, 100000, 1000000]]
mp_err = [d_gil[f"pingpong_multiprocess_{num_bytes}b"][1] * 1000.0 for num_bytes in [1, 1000, 100000, 1000000]]

In [48]:
fig = go.Figure()

fig.add_bar(
    name="Subinterpreter",
    x=msg_sizes,
    y=subint_mean,
    error_y=dict(type="data", array=subint_err, visible=True),
    width=0.25,
    textposition="outside"
)

fig.add_bar(
    name="Multiprocess",
    x=msg_sizes,
    y=mp_mean,
    error_y=dict(type="data", array=mp_err, visible=True),
    width=0.25,
    textposition="outside"
)

fig.update_layout(
    width=1200,
    xaxis=dict(tickfont=dict(size=18)),
    xaxis_title="Msg Size (Bytes)",
    yaxis_title="Time (ms)",
    barmode="group",
    bargap=0.2,
    legend_title_text="Method",
    template="plotly_white"
)

fig.show()

In [49]:
fig = go.Figure()

fig.add_bar(
    name="Thread noGIL",
    x=msg_sizes,
    y=nogil_mean,
    error_y=dict(type="data", array=nogil_err, visible=True),
    width=0.25,
    textposition="outside"
)

fig.add_bar(
    name="Thread with GIL",
    x=msg_sizes,
    y=gil_mean,
    error_y=dict(type="data", array=gil_err, visible=True),
    width=0.25,
    textposition="outside"
)

fig.update_layout(
    width=1200,
    xaxis=dict(tickfont=dict(size=18)),
    xaxis_title="Msg Size (Bytes)",
    yaxis_title="Time (ms)",
    barmode="group",
    bargap=0.2,
    legend_title_text="Method",
    template="plotly_white"
)

fig.show()

### Test 3: Definite integral

In [4]:
!python3.14t ./Tests/integral.py --quiet --rigorous -o ./Data/integral_nogil.json

integral_threads_w1: Mean +- std dev: 408 ms +- 22 ms
integral_mp_w1: Mean +- std dev: 577 ms +- 16 ms
integral_subinterp_w1: Mean +- std dev: 585 ms +- 16 ms
integral_threads_w2: Mean +- std dev: 217 ms +- 19 ms
integral_mp_w2: Mean +- std dev: 373 ms +- 23 ms
integral_subinterp_w2: Mean +- std dev: 423 ms +- 24 ms
integral_threads_w3: Mean +- std dev: 161 ms +- 20 ms
integral_mp_w3: Mean +- std dev: 311 ms +- 19 ms
integral_subinterp_w3: Mean +- std dev: 417 ms +- 27 ms
integral_threads_w4: Mean +- std dev: 126 ms +- 17 ms
integral_mp_w4: Mean +- std dev: 282 ms +- 22 ms
integral_subinterp_w4: Mean +- std dev: 471 ms +- 50 ms
integral_threads_w5: Mean +- std dev: 112 ms +- 12 ms
integral_mp_w5: Mean +- std dev: 279 ms +- 21 ms
integral_subinterp_w5: Mean +- std dev: 495 ms +- 38 ms
integral_threads_w6: Mean +- std dev: 98.4 ms +- 9.8 ms
integral_mp_w6: Mean +- std dev: 266 ms +- 17 ms
integral_subinterp_w6: Mean +- std dev: 567 ms +- 40 ms
integral_threads_w7: Mean +- std dev: 88.5 m

In [5]:
!python3.14 ./Tests/integral.py --quiet --rigorous -o ./Data/integral_gil.json

integral_threads_w1: Mean +- std dev: 350 ms +- 20 ms
integral_mp_w1: Mean +- std dev: 485 ms +- 59 ms
integral_subinterp_w1: Mean +- std dev: 483 ms +- 43 ms
integral_threads_w2: Mean +- std dev: 387 ms +- 16 ms
integral_mp_w2: Mean +- std dev: 356 ms +- 53 ms
integral_subinterp_w2: Mean +- std dev: 317 ms +- 15 ms
integral_threads_w3: Mean +- std dev: 384 ms +- 5 ms
integral_mp_w3: Mean +- std dev: 266 ms +- 24 ms
integral_subinterp_w3: Mean +- std dev: 282 ms +- 23 ms
integral_threads_w4: Mean +- std dev: 383 ms +- 6 ms
integral_mp_w4: Mean +- std dev: 229 ms +- 22 ms
integral_subinterp_w4: Mean +- std dev: 261 ms +- 18 ms
integral_threads_w5: Mean +- std dev: 386 ms +- 7 ms
integral_mp_w5: Mean +- std dev: 214 ms +- 13 ms
integral_subinterp_w5: Mean +- std dev: 273 ms +- 10 ms
integral_threads_w6: Mean +- std dev: 391 ms +- 20 ms
integral_mp_w6: Mean +- std dev: 209 ms +- 11 ms
integral_subinterp_w6: Mean +- std dev: 280 ms +- 9 ms
integral_threads_w7: Mean +- std dev: 379 ms +- 14

In [43]:
workers_num = [i for i in range(1, 16 + 1)]

d_nogil = get_result_data("./Data/integral_nogil.json")
d_gil = get_result_data("./Data/integral_gil.json")

data_mean = {
    "Threads noGIL":        [d_nogil[f"integral_threads_w{w}"][0] * 1000.0 for w in range(1, 16 + 1)],
    "Threads with GIL":     [d_gil[f"integral_threads_w{w}"][0] * 1000.0 for w in range(1, 16 + 1)],
    "Subinterpreters":      [d_gil[f"integral_subinterp_w{w}"][0] * 1000.0 for w in range(1, 16 + 1)],
    "Multiprocess":         [d_gil[f"integral_mp_w{w}"][0] * 1000.0 for w in range(1, 16 + 1)],
}

In [44]:
fig = go.Figure()

for name, y in data_mean.items():
    fig.add_trace(go.Scatter(
        x=workers_num,
        y=y,
        name=name,
        mode="lines+markers",
        marker=dict(size=10, symbol="circle"),
        line=dict(width=3),
        hovertemplate="x=%{x}<br>y=%{y:.3f} ms<extra></extra>"
    ))

fig.update_xaxes(
    tickmode="array",
    tickvals=workers_num,
    ticktext=[str(val) for val in workers_num]
)

fig.update_layout(
    width=1200,
    xaxis=dict(tickfont=dict(size=16)),
    xaxis_title="Number of workers",
    yaxis_title="Time (ms)",
    barmode="group",
    bargap=0.2,
    legend_title_text="Method",
    template="plotly_white"
)

fig.show()

### Test 4: IO-bound (https://gist.github.com/sobolevn/149a461c629f6e03aef3a772c7422b2e)

In [6]:
!python3.14t ./Tests/external_test_io.py --quiet --rigorous -o ./Data/io_nogil.json

Regular: Mean +- std dev: 1.81 sec +- 0.14 sec
Threading: Mean +- std dev: 551 ms +- 214 ms
Multiprocessing: Mean +- std dev: 568 ms +- 67 ms
Subinterpreters: Mean +- std dev: 986 ms +- 88 ms


In [7]:
!python3.14 ./Tests/external_test_io.py --quiet --rigorous -o ./Data/io_gil.json

Regular: Mean +- std dev: 1.88 sec +- 0.11 sec
Threading: Mean +- std dev: 588 ms +- 212 ms
Multiprocessing: Mean +- std dev: 628 ms +- 136 ms
Subinterpreters: Mean +- std dev: 1.05 sec +- 0.09 sec


In [34]:
d_nogil = get_result_data("./Data/io_nogil.json")
d_gil = get_result_data("./Data/io_gil.json")

names = ["Regular", "Thread noGIL", "Thread with GIL", "Subinterpreter", "Multiprocess"]

data_mean = [
    d_gil["Regular"][0], 
    d_nogil["Threading"][0], 
    d_gil["Threading"][0], 
    d_gil["Subinterpreters"][0], 
    d_gil["Multiprocessing"][0]
]
data_err = [
    d_gil["Regular"][1], 
    d_nogil["Threading"][1], 
    d_gil["Threading"][1], 
    d_gil["Subinterpreters"][1], 
    d_gil["Multiprocessing"][1]
]

In [35]:
fig = go.Figure()

fig.add_bar(
    x=names,
    y=data_mean,
    error_y=dict(type="data", array=data_err, visible=True),
    width=0.25,
    textposition="outside"
)

fig.update_layout(
    width=1200,
    xaxis=dict(tickfont=dict(size=18)),
    yaxis_title="Time (sec)",
    barmode="group",
    bargap=0.5,
    legend_title_text="Method",
    template="plotly_white"
)

fig.show()

### Test 5: CPU-bound (https://gist.github.com/sobolevn/149a461c629f6e03aef3a772c7422b2e)

In [3]:
!python3.14t ./Tests/external_test_cpu.py --quiet --rigorous -o ./Data/cpu_nogil.json

Regular: Mean +- std dev: 89.3 ms +- 1.4 ms
Threading: Mean +- std dev: 28.8 ms +- 0.6 ms
Multiprocessing: Mean +- std dev: 265 ms +- 9 ms
Subinterpreters: Mean +- std dev: 424 ms +- 87 ms


In [4]:
!python3.14 ./Tests/external_test_cpu.py --quiet --rigorous -o ./Data/cpu_gil.json

Regular: Mean +- std dev: 123 ms +- 24 ms
Threading: Mean +- std dev: 136 ms +- 19 ms
Multiprocessing: Mean +- std dev: 361 ms +- 58 ms
Subinterpreters: Mean +- std dev: 141 ms +- 17 ms


In [5]:
d_nogil = get_result_data("./Data/cpu_nogil.json")
d_gil = get_result_data("./Data/cpu_gil.json")

names = ["Regular", "Thread noGIL", "Thread with GIL", "Subinterpreter", "Multiprocess"]

data_mean = [
    d_gil["Regular"][0], 
    d_nogil["Threading"][0], 
    d_gil["Threading"][0], 
    d_gil["Subinterpreters"][0], 
    d_gil["Multiprocessing"][0]
]
data_err = [
    d_gil["Regular"][1], 
    d_nogil["Threading"][1], 
    d_gil["Threading"][1], 
    d_gil["Subinterpreters"][1], 
    d_gil["Multiprocessing"][1]
]

In [6]:
fig = go.Figure()

fig.add_bar(
    x=names,
    y=data_mean,
    error_y=dict(type="data", array=data_err, visible=True),
    width=0.25,
    textposition="outside"
)

fig.update_layout(
    width=1200,
    xaxis=dict(tickfont=dict(size=18)),
    yaxis_title="Time (sec)",
    barmode="group",
    bargap=0.5,
    legend_title_text="Method",
    template="plotly_white"
)

fig.show()